# Smart Lender — Loan Approval Prediction using ML

This notebook covers Epics 1–4 of the Smart Lender project: importing and reading the dataset, visualizing and analysing the data, data pre-processing, and model building.

## Epic 2: Visualizing and Analysing the Data
### Importing the Libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
plt.style.use('fivethirtyeight')

import sklearn
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split, cross_val_score

from imblearn.over_sampling import SMOTE

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

### Import and Read Dataset
The dataset is in `.csv` format. The `read_csv()` function is used to load it by passing the file path as a parameter.

In [ ]:
#importing the dataset
data = pd.read_csv('../Dataset/loan_prediction.csv')
data.head()

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

### Univariate Analysis
Distribution plots (`distplot`/`histplot`) for continuous variables, count plots (`countplot`) for categorical variables.

In [ ]:
#plotting the numerical distributions
plt.figure(figsize=(12,5))
plt.subplot(121)
sns.histplot(data['ApplicantIncome'], color='r', kde=True)
plt.subplot(122)
sns.histplot(data['Credit_History'].dropna(), kde=True)
plt.show()

In [ ]:
#plotting the count plot for categorical columns
plt.figure(figsize=(18,4))
cat_cols = ['Gender','Married','Dependents']
for i, col in enumerate(cat_cols):
    plt.subplot(1, len(cat_cols), i+1)
    sns.countplot(x=data[col])
plt.tight_layout()
plt.show()

### Bivariate Analysis
Count plots are used to examine relationships between pairs of variables (e.g. Gender vs Married, Education vs Self_Employed, Property_Area vs Loan_Amount_Term).

In [ ]:
#visualising two columns against each other
plt.figure(figsize=(20,5))
plt.subplot(131)
sns.countplot(x=data['Married'], hue=data['Gender'])
plt.subplot(132)
sns.countplot(x=data['Self_Employed'], hue=data['Education'])
plt.subplot(133)
sns.countplot(x=data['Property_Area'], hue=data['Loan_Amount_Term'])
plt.tight_layout()
plt.show()

### Multivariate Analysis
`swarmplot` from seaborn is used to examine relationships between multiple features simultaneously.

In [ ]:
plt.figure(figsize=(10,5))
sns.swarmplot(x='Education', y='ApplicantIncome', hue='Loan_Status', data=data)
plt.show()

## Epic 3: Data Pre-processing
### Handling Missing Values
Missing values in numerical features are replaced with the mean; missing values in categorical features are replaced with the mode.

In [ ]:
for col in ['Gender','Married','Dependents','Self_Employed','Credit_History','Loan_Amount_Term']:
    data[col] = data[col].fillna(data[col].mode()[0])

data['LoanAmount'] = data['LoanAmount'].fillna(data['LoanAmount'].mean())

data.isnull().sum()

### Handling Categorical Values
Categorical columns are converted into numerical values using encoding so the data is compatible with machine learning algorithms.

In [ ]:
data['Gender'] = data['Gender'].map({'Male':1,'Female':0})
data['Married'] = data['Married'].map({'Yes':1,'No':0})
data['Education'] = data['Education'].map({'Graduate':1,'Not Graduate':0})
data['Self_Employed'] = data['Self_Employed'].map({'Yes':1,'No':0})
data['Dependents'] = data['Dependents'].replace('3+', 3).astype(int)
data['Property_Area'] = data['Property_Area'].map({'Rural':0,'Semiurban':1,'Urban':2})
data['Loan_Status'] = data['Loan_Status'].map({'Y':1,'N':0})

data.head()

### Splitting Features and Target

In [ ]:
X = data.drop(['Loan_ID','Loan_Status'], axis=1)
y = data['Loan_Status']

y.value_counts()

### Handling the Imbalanced Dataset
SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic samples for the minority class using K-Nearest Neighbors, so the model does not become biased toward the majority class.

In [ ]:
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

y_res.value_counts()

### Feature Scaling
Feature scaling is applied to the input features only (not the target), since models like KNN rely on distance-based calculations.

In [ ]:
scale = StandardScaler()
X_scaled = scale.fit_transform(X_res)
X_scaled = pd.DataFrame(X_scaled, columns=X_res.columns)
X_scaled.head()

### Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_res, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

## Epic 4: Model Building
Four models are trained and compared: Decision Tree, Random Forest, KNN, and Gradient Boosting (referred to here as "xgboost").

In [ ]:
def decisionTree(X_train, X_test, y_train, y_test):
    dt = DecisionTreeClassifier(random_state=42)
    dt.fit(X_train, y_train)
    pred = dt.predict(X_test)
    print('Accuracy:', accuracy_score(y_test, pred))
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred))
    return dt

dt_model = decisionTree(X_train, X_test, y_train, y_test)

In [ ]:
def randomForest(X_train, X_test, y_train, y_test):
    rf = RandomForestClassifier(random_state=42)
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)
    print('Accuracy:', accuracy_score(y_test, pred))
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred))
    return rf

rf_model = randomForest(X_train, X_test, y_train, y_test)

In [ ]:
def KNN(X_train, X_test, y_train, y_test):
    knn = KNeighborsClassifier()
    knn.fit(X_train, y_train)
    pred = knn.predict(X_test)
    print('Accuracy:', accuracy_score(y_test, pred))
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred))
    return knn

knn_model = KNN(X_train, X_test, y_train, y_test)

In [ ]:
def xgboost(X_train, X_test, y_train, y_test):
    xgb = GradientBoostingClassifier(random_state=42)
    xgb.fit(X_train, y_train)
    pred = xgb.predict(X_test)
    print('Accuracy:', accuracy_score(y_test, pred))
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred))
    return xgb

xgb_model = xgboost(X_train, X_test, y_train, y_test)

### Cross Validation
5-fold cross-validation is used to verify consistent model performance across different data splits.

In [ ]:
models = {
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'KNN': knn_model,
    'XGBoost (GradientBoosting)': xgb_model
}

cv_scores = {}
for name, model in models.items():
    score = cross_val_score(model, X_scaled, y_res, cv=5).mean()
    cv_scores[name] = score
    print(f'{name}: {score:.4f}')

best_model_name = max(cv_scores, key=cv_scores.get)
print('\nBest Model:', best_model_name)

### Model Storage
The best-performing model is saved with `pickle` as `rdf.pkl` for use in the Flask application, along with the fitted `StandardScaler` as `scale1.pkl`.

In [ ]:
best_model = models[best_model_name]

pickle.dump(best_model, open('../Flask/rdf.pkl', 'wb'))
pickle.dump(scale, open('../Flask/scale1.pkl', 'wb'))

print('Model and scaler saved successfully.')